In [1]:
# =========================
# nb_run_all
# =========================

# ---------- Parameters ----------
period = "2026-03"   # YYYY-MM

# ---------- Imports ----------
from datetime import datetime, timezone
from pyspark.sql import functions as F

# ---------- Run identity ----------
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
print(f"[nb_run_all] START period={period} run_id={run_id}")

# ---------- Helpers ----------
def q(s: str) -> str:
    return (s or "").replace("'", "''")

def audit_orchestrator(status, message):
    # Uses run_audit_log contract; dfm_id reserved as 'orchestrator'
    spark.sql(f"""
        INSERT INTO run_audit_log
        (run_id, period, dfm_id, files_processed, rows_ingested, parse_errors_count, drift_events_count, status, started_at, completed_at)
        VALUES
        ('{q(run_id)}','{q(period)}','orchestrator',0,0,0,0,'{q(status)}',current_timestamp(),current_timestamp())
    """)
    print(f"[orchestrator] {status}: {message}")

def run_notebook(nb_name, args, timeout_sec=3600):
    print(f"\n--- Running {nb_name} with args={args}")
    try:
        res = mssparkutils.notebook.run(nb_name, timeout_sec, args)
        res_text = (res or "").strip()
        print(f"--- {nb_name} completed with result: {res_text}")
        return "OK", res_text
    except Exception as ex:
        err = f"{type(ex).__name__}: {str(ex)}"
        print(f"--- {nb_name} FAILED: {err}")
        return "FAILED", err

# ---------- Execution plan ----------
ingest_jobs = [
    ("nb_ingest_wh_ireland", "wh_ireland"),
    ("nb_ingest_pershing", "pershing"),
    ("nb_ingest_castlebay", "castlebay"),
    ("nb_ingest_brown_shipley", "brown_shipley"),
]

post_jobs = [
    "nb_validate",
    "nb_aggregate",
    "nb_reports",
]

# ---------- Start audit ----------
audit_orchestrator("RUNNING", "Orchestration started")

# ---------- Run ingestion notebooks ----------
ingest_results = []
any_hard_fail = False

for nb, dfm in ingest_jobs:
    status, detail = run_notebook(
        nb_name=nb,
        args={
            "period": period,
            "run_id": run_id
        },
        timeout_sec=3600
    )
    ingest_results.append((nb, dfm, status, detail))
    if status == "FAILED":
        any_hard_fail = True
        # continue to next DFM for fault isolation

# ---------- Decide whether to continue ----------
# Continue to post-processing if at least one ingest notebook did not hard-fail.
ingest_ok_or_partial = [r for r in ingest_results if r[2] != "FAILED"]

if len(ingest_ok_or_partial) == 0:
    audit_orchestrator("FAILED", "All ingest notebooks failed; skipping validate/aggregate/reports")
    mssparkutils.notebook.exit(f"FAILED|run_id={run_id}|reason=all_ingests_failed")

# ---------- Run post-processing notebooks ----------
post_results = []
for nb in post_jobs:
    status, detail = run_notebook(
        nb_name=nb,
        args={
            "period": period,
            "run_id": run_id
        },
        timeout_sec=3600
    )
    post_results.append((nb, status, detail))
    if status == "FAILED":
        any_hard_fail = True
        # stop post chain on first hard failure
        break

# ---------- Final status ----------
ingest_failed_count = len([r for r in ingest_results if r[2] == "FAILED"])
post_failed_count = len([r for r in post_results if r[1] == "FAILED"])

if any_hard_fail:
    final_status = "PARTIAL" if len(ingest_ok_or_partial) > 0 else "FAILED"
else:
    final_status = "OK"

summary = (
    f"run_id={run_id}; "
    f"ingest_failed={ingest_failed_count}/{len(ingest_results)}; "
    f"post_failed={post_failed_count}/{len(post_jobs)}"
)

audit_orchestrator(final_status, summary)

print("\n=== RUN SUMMARY ===")
print(summary)
print("Ingest results:")
for r in ingest_results:
    print(r)
print("Post results:")
for r in post_results:
    print(r)

mssparkutils.notebook.exit(f"{final_status}|run_id={run_id}")

StatementMeta(, 192f92f6-78bb-4563-be75-a95dda87885b, 3, Finished, Available, Finished, False)

[nb_run_all] START period=2026-03 run_id=20260302T220333Z
[orchestrator] RUNNING: Orchestration started

--- Running nb_ingest_wh_ireland with args={'period': '2026-03', 'run_id': '20260302T220333Z'}


--- nb_ingest_wh_ireland completed with result: NO_FILES

--- Running nb_ingest_pershing with args={'period': '2026-03', 'run_id': '20260302T220333Z'}


--- nb_ingest_pershing FAILED: Py4JJavaError: An error occurred while calling o5219.throwExceptionIfHave.
: com.microsoft.spark.notebook.msutils.NotebookExecutionException: [AMBIGUOUS_REFERENCE] Reference `source_file` is ambiguous, could be: [`source_file`, `source_file`].
---------------------------------------------------------------------------AnalysisException                         Traceback (most recent call last)Cell In[4], line 362
    330 good = good.withColumn(
    331     "source_row_id",
    332     F.sha2(
   (...)
    344     )
    345 )
    347 # ---------- Canonical projection ----------
    348 canonical = (
    349     good
    350     .withColumn("period", F.lit(period))
    351     .withColumn("run_id", F.lit(run_id))
    352     .withColumn("dfm_id", F.lit(dfm_id))
    353     .withColumn("dfm_name", F.lit(dfm_name))
    354     .withColumn("source_file", F.col("source_file_out"))
    355     .withColumn("source_sheet", F.lit(None).cast("string"))
    356     .wi

--- nb_ingest_castlebay FAILED: Py4JJavaError: An error occurred while calling o5229.throwExceptionIfHave.
: com.microsoft.spark.notebook.msutils.NotebookExecutionException: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 85f41908-fd38-4b99-b667-d1fb8262c0c6).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- period: string (nullable = true)
-- run_id: string (nullable = true)
-- dfm_id: string (nullable = true)
-- source_file: string (nullable = true)
-- row_number: long (nullable = true)
-- column_name: string (nullable = true)
-- raw_value: string (nullable = true)
-- error_type: string (nullable = true)
-- error_message: string (nullable = true)
-- event_ts: timestamp (nul

--- nb_ingest_brown_shipley FAILED: Py4JJavaError: An error occurred while calling o5239.throwExceptionIfHave.
: com.microsoft.spark.notebook.msutils.NotebookExecutionException: [AMBIGUOUS_REFERENCE] Reference `source_file` is ambiguous, could be: [`source_file`, `source_file`].
---------------------------------------------------------------------------AnalysisException                         Traceback (most recent call last)Cell In[4], line 388
    355 good = good.withColumn(
    356     "source_row_id",
    357     F.sha2(
   (...)
    371     )
    372 )
    374 # ---------- Canonical projection ----------
    375 canonical = (
    376     good
    377     .withColumn("period", F.lit(period))
    378     .withColumn("run_id", F.lit(run_id))
    379     .withColumn("dfm_id", F.lit(dfm_id))
    380     .withColumn("dfm_name", F.lit(dfm_name))
    381     .withColumn("source_file", F.col("source_file_out"))
    382     .withColumn("source_sheet", F.lit(None).cast("string"))
    383   

--- nb_validate completed with result: NO_DATA

--- Running nb_aggregate with args={'period': '2026-03', 'run_id': '20260302T220333Z'}


--- nb_aggregate completed with result: NO_DATA

--- Running nb_reports with args={'period': '2026-03', 'run_id': '20260302T220333Z'}


--- nb_reports completed with result: NO_DATA
[orchestrator] PARTIAL: run_id=20260302T220333Z; ingest_failed=3/4; post_failed=0/3

=== RUN SUMMARY ===
run_id=20260302T220333Z; ingest_failed=3/4; post_failed=0/3
Ingest results:
('nb_ingest_wh_ireland', 'wh_ireland', 'OK', 'NO_FILES')
('nb_ingest_pershing', 'pershing', 'FAILED', 'Py4JJavaError: An error occurred while calling o5219.throwExceptionIfHave.\n: com.microsoft.spark.notebook.msutils.NotebookExecutionException: [AMBIGUOUS_REFERENCE] Reference `source_file` is ambiguous, could be: [`source_file`, `source_file`].\n---------------------------------------------------------------------------AnalysisException                         Traceback (most recent call last)Cell In[4], line 362\n    330 good = good.withColumn(\n    331     "source_row_id",\n    332     F.sha2(\n   (...)\n    344     )\n    345 )\n    347 # ---------- Canonical projection ----------\n    348 canonical = (\n    349     good\n    350     .withColumn("period", F.l

In [ ]:
# 1) audit rows for this run
spark.sql(f"""
SELECT run_id, period, dfm_id, status, files_processed, rows_ingested, parse_errors_count, started_at, completed_at
FROM run_audit_log
WHERE run_id = '{run_id}'
ORDER BY dfm_id, completed_at
""").show(truncate=False)

# 2) canonical row counts by DFM
spark.sql(f"""
SELECT dfm_id, COUNT(*) AS rows
FROM canonical_holdings
WHERE run_id = '{run_id}'
GROUP BY dfm_id
ORDER BY dfm_id
""").show()

StatementMeta(, 192f92f6-78bb-4563-be75-a95dda87885b, -1, Cancelled, , Cancelled, True)